In [ ]:
import os

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = OpenAI(
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
)

MODEL = "deepseek-chat"

In [ ]:
class StatelessChatBot:
    def __init__(self, client, model):
        self.model = model
        self.client = client

    def chat(self, user_input):
        messages = [{"role": "user", "content": user_input}]
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
        )
        return response.choices[0].message.content

In [ ]:
bot = StatelessChatBot(client, MODEL)
print("第一轮测试：", bot.chat("我是詹姆斯，从事大模型相关工作。"))

In [ ]:
print("第二轮测试：", bot.chat("你还记得我的名字和职业吗？"))

In [ ]:
class MemoryChatBot:
    def __init__(self, client, model):
        self.model = model
        self.client = client
        self.messages: list[dict[str, str]] = []

    def add_message(self, role, content):
        self.messages.append({"role": role, "content": content})

    def chat(self, user_input):
        self.add_message("user", user_input)  # 将用户输入添加到消息列表中
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
        )
        reply = response.choices[0].message.content
        self.add_message("assistant", reply) # 将模型回复添加到消息列表中
        return reply

In [ ]:
bot = MemoryChatBot(client, MODEL)
print("第一轮测试：", bot.chat("我是詹姆斯，从事大模型相关工作。"))

In [ ]:
print("第二轮测试：", bot.chat("你还记得我的名字和职业吗？"))

In [ ]:
bot.messages

In [ ]:
from dataclasses import dataclass, field
import time

@dataclass
class Session:
    title: str = ""
    create_time:int = field(default_factory=lambda: int(time.time()))
    update_time:int = field(default_factory=lambda: int(time.time()))
    messages: list[dict[str, str]] = field(default_factory=list)
    compress_content: str = ""

    def touch(self) -> None:
        self.update_time = int(time.time())

In [ ]:
from dataclasses import asdict
import json
from pathlib import Path


class JsonSessionStore:
    def __init__(self, base_dir:str="./sessions"):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)

    def _get_path(self, session_id:str) -> Path:
        return self.base_dir / f"{session_id}.json"

    def save(self, session_id:str, session: Session) -> None:
        session.touch()
        path = self._get_path(session_id)
        with path.open("w", encoding="utf-8") as f:
            json.dump(asdict(session), f, ensure_ascii=False, indent=2)

    def load(self, session_id:str) -> Session:
        path = self._get_path(session_id)
        if not path.exists():
            return Session()
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
            return Session(**data)

In [ ]:
store = JsonSessionStore()
session_id = "pk_001"
session = store.load(session_id)
print("加载的会话消息:", session.messages)

In [ ]:
session.messages.append({"role": "user", "content": "我是詹姆斯，从事大模型相关工作。"})
store.save(session_id, session)

In [ ]:
load_msg = store.load(session_id)
load_msg

In [ ]:
session.messages.append({"role": "assistant", "content": "你好，詹姆斯！很高兴认识你！作为大模型领域的从业者，你的工作一定充满了挑战与前沿探索"})
store.save(session_id, session)

In [ ]:
session.messages

In [ ]:
from typing import Any

@dataclass
class ContextWindowBuilder:
    max_history:int = 20
    verbose:bool = False

    def build(self, session:Any) -> list[dict[str, str]]:
        
        msgs: list[dict[str, str]] = []
        recent_messages = session.messages[-self.max_history:] # 获取最近的N条消息记录

        if self.verbose:
            for msg in recent_messages:
                print(f"{msg['role']}  : {msg['content']}")


        msgs.extend(recent_messages)
        return msgs


In [ ]:
builder = ContextWindowBuilder(max_history=20, verbose=True)
session.messages = [
    {"role":"user", "content": f"消息{i}"} for i in range(30)
]
result = builder.build(session)
print(len(result))

In [ ]:
from dataclasses import dataclass
from typing import Any


@dataclass
class RollingSummaryCompressor:
    client: Any
    model: str
    min_messages: int = 4  # 至少保留最近4条消息参与压缩
    keep_ratio: float = 0.5
    prefix: str = "[以下是之前对话的摘要]"

    def compress(self, session: Any) -> Any:
        """
        滚动压缩会话历史：
        - 压缩前半段历史
        - 保留后半段继续参与上下文
        - 新摘要覆盖旧摘要
        """
        messages = session.messages

        if len(messages) < self.min_messages:
            return session

        # 取前 50% 作为待压缩部分（至少 4 条）
        split_index = max(self.min_messages, int(len(messages) * self.keep_ratio)) # 计算分割索引
        to_compress = messages[:split_index] # 待压缩历史
        to_keep = messages[split_index:]  # 保留后半段历史

        summary_input = self._build_summary_input(
            session.compress_content,
            to_compress
        )

        summary = self._summarize(summary_input)

        session.compress_content = summary
        session.messages = to_keep
        return session

    # 滚动摘要：把已有摘要 + 新消息一起压缩成统一摘要   
    def _build_summary_input(
        self,
        existing_summary: str,
        messages: list[dict[str, str]]
    ) -> str:
        history_text = "\n".join(
            f"{m['role'].upper()}: {m['content']}"
            for m in messages
        )

        if existing_summary:
            return f"[之前摘要]\n{existing_summary}\n\n[新增对话]\n{history_text}"

        return history_text

    def _summarize(self, content: str) -> str:
        prompt = f"""
请将以下内容压缩成统一摘要，保留关键事实：
- 用户身份
- 技术背景
- 已完成任务
- 未完成任务
- 重要决策

要求：
- 使用第三人称（用户 / 助手）
- 100~200字
- 必须以「{self.prefix}」开头

内容：
{content}
"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            timeout=30
        )

        return response.choices[0].message.content

In [ ]:
compressor = RollingSummaryCompressor(client, MODEL)

session = Session(
    messages=[
        {"role": "user", "content": "我准备装修新房，预算15万"},
        {"role": "assistant", "content": "好的，我可以帮你规划装修预算和风格。"},
        {"role": "user", "content": "我喜欢现代简约风，客厅想做无主灯设计"},
        {"role": "assistant", "content": "现代简约很适合无主灯，建议搭配磁吸轨道灯。"},
        {"role": "user", "content": "厨房我想用洗碗机和嵌入式冰箱"},
        {"role": "assistant", "content": "好的，这样预算里需要预留约2万元电器费用。"},
    ]
)

print(f"压缩前：{len(session.messages)}")  # 6
session = compressor.compress(session)
print(f"压缩后：{len(session.messages)}")
print(session.compress_content)
for msg in session.messages:
    print(f"\t{msg['role']} -> {msg['content']}")